# 00 — Download, Setup & ERA5 Data Audit
## เริ่มต้นการเรียนรู้ข้อมูล Reanalysis ด้วย ERA5, NetCDF และ Xarray

**กลุ่มผู้เรียน:** นิสิตปริญญาตรีชั้นปีที่ 3–4 ด้านสิ่งแวดล้อม ภูมิศาสตร์ และภูมิสารสนเทศ  
**Platform:** Google Colab + Google Drive  
**Repository:** `nattaponm/Teaching_ERA5_Reanalysis_GIS`

Notebook นี้เป็น **จุดเริ่มต้นของนิสิต** สำหรับชุดแบบเรียน ERA5 Reanalysis and GIS

```text
GitHub teaching data
        ↓
Google Drive
        ↓
Open NetCDF with Xarray
        ↓
Audit dimensions / coordinates / variables
        ↓
Audit time / pressure / space / units
        ↓
COURSE DATA READY
```

> Notebook 00A เป็น Instructor notebook สำหรับเตรียมข้อมูล  
> Notebook 00 นี้เป็น Student starting notebook

# Learning Objectives

เมื่อจบบทนี้ นิสิตควรสามารถ:

1. อธิบายว่า ERA5 เป็น **reanalysis** ไม่ใช่ direct observation
2. แยก observation, forecast และ reanalysis ได้
3. อธิบายโครงสร้างข้อมูล NetCDF แบบหลายมิติ
4. เข้าใจ `Dataset`, `DataArray`, `dimension`, `coordinate`, `variable`, `attribute`
5. อธิบาย atmospheric data cube
6. ดาวน์โหลด teaching datasets จาก GitHub อัตโนมัติ
7. เก็บข้อมูลไว้ใน Google Drive เพื่อใช้ต่อใน Notebook 01–09
8. เปิด NetCDF ด้วย Xarray
9. ตรวจ time coverage, pressure levels, spatial domain และ grid spacing
10. เข้าใจข้อจำกัดของ teaching datasets ก่อนวิเคราะห์

# 1. Reanalysis คืออะไร?

```text
Observation
→ สิ่งที่เครื่องมือวัดจริง

Forecast
→ แบบจำลองคาดการณ์อนาคต

Reanalysis
→ การประมวลสภาพบรรยากาศย้อนหลัง
  โดยใช้ model + observations + data assimilation
```

ERA5 เป็น atmospheric reanalysis

> **ERA5 ≠ direct observation**

# 2. Atmospheric Data Cube

ตัวแปรหนึ่ง เช่น temperature สามารถเขียนเป็น:

$$
T = T(t,p,\phi,\lambda)
$$

โดย:

- $t$ = time
- $p$ = pressure level
- $\phi$ = latitude
- $\lambda$ = longitude

ตัวอย่าง:

```text
EVENT
(time, level, latitude, longitude)

SPATIOTEMPORAL
(time, latitude, longitude)

MONTHLY_BASELINE
(time, latitude, longitude)
```

# 3. Teaching datasets

ข้อมูลอยู่ที่:

`https://github.com/nattaponm/Teaching_ERA5_Reanalysis_GIS/tree/main/00_prepared_data`

ไฟล์หลัก:

```text
era5_noul_20111002_05_SEAsia_pressure_levels_0.5deg.nc
era5_u850_6hourly_20110901_20111031_Asia_0.5deg.nc
era5_850hPa_JanJul_monthlymeans_1981_2020_Asia_0.5deg.nc
```

บทบาท:

```text
EVENT
→ pressure levels / synoptic analysis

SPATIOTEMPORAL
→ 6-hourly 850-hPa zonal wind / Hovmöller

MONTHLY_BASELINE
→ Jan/Jul monthly means / climatology / anomaly
```

# 4. Scientific caveats ก่อนเริ่ม

```text
ERA5 ≠ direct observation
Grid spacing ≠ effective physical resolution
Monthly mean ≠ climatological mean
00-UTC snapshot ≠ daily mean
850 hPa ≠ above-ground atmosphere everywhere
January/July baseline ≠ October climatology
```

# 5. ติดตั้งไลบรารี

In [1]:
# CELL 1 — Install/check libraries
import sys, subprocess

PACKAGES = [
    "xarray>=2025.1",
    "netCDF4>=1.7",
    "h5netcdf>=1.4",
    "dask[array]>=2025.1",
    "numpy>=1.26",
    "pandas>=2.2",
    "requests>=2.31",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--upgrade-strategy", "only-if-needed",
    *PACKAGES
])

print("Libraries ready.")

Libraries ready.


# 6. Mount Google Drive

In [2]:
# CELL 2 — Mount Google Drive
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")

print("Google Drive mounted:", DRIVE_ROOT)

Mounted at /content/drive
Google Drive mounted: /content/drive/MyDrive


# 7. Import libraries

In [3]:
# CELL 3 — Imports
from __future__ import annotations

import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import requests
from IPython.display import display

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Xarray:", xr.__version__)

Python: 3.13.15
NumPy: 2.1.3
pandas: 2.2.3
Xarray: 2025.12.0


# 8. Course workspace

```text
MyDrive/
└── Teaching_ERA5_Reanalysis_GIS/
    ├── 00_source_github/
    ├── 01_data/
    ├── 02_metadata/
    ├── 03_output/
    ├── 04_figures/
    └── 99_logs/
```

In [4]:
# CELL 4 — Create course folders
COURSE_ROOT = DRIVE_ROOT / "Teaching_ERA5_Reanalysis_GIS"

SOURCE_GITHUB_DIR = COURSE_ROOT / "00_source_github"
DATA_DIR = COURSE_ROOT / "01_data"
METADATA_DIR = COURSE_ROOT / "02_metadata"
OUTPUT_DIR = COURSE_ROOT / "03_output"
FIGURE_DIR = COURSE_ROOT / "04_figures"
LOG_DIR = COURSE_ROOT / "99_logs"

for folder in [
    SOURCE_GITHUB_DIR, DATA_DIR, METADATA_DIR,
    OUTPUT_DIR, FIGURE_DIR, LOG_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Course root:", COURSE_ROOT)

Course root: /content/drive/MyDrive/Teaching_ERA5_Reanalysis_GIS


# 9. กำหนด GitHub source

In [5]:
# CELL 5 — GitHub file definitions
REPO_OWNER = "nattaponm"
REPO_NAME = "Teaching_ERA5_Reanalysis_GIS"
REPO_BRANCH = "main"
REPO_DATA_FOLDER = "00_prepared_data"

RAW_BASE = (
    f"https://raw.githubusercontent.com/"
    f"{REPO_OWNER}/{REPO_NAME}/{REPO_BRANCH}/{REPO_DATA_FOLDER}"
)

TEACHING_FILES = {
    "EVENT": "era5_noul_20111002_05_SEAsia_pressure_levels_0.5deg.nc",
    "SPATIOTEMPORAL": "era5_u850_6hourly_20110901_20111031_Asia_0.5deg.nc",
    "MONTHLY_BASELINE": "era5_850hPa_JanJul_monthlymeans_1981_2020_Asia_0.5deg.nc",
}

for key, filename in TEACHING_FILES.items():
    print(key, "→", filename)

EVENT → era5_noul_20111002_05_SEAsia_pressure_levels_0.5deg.nc
SPATIOTEMPORAL → era5_u850_6hourly_20110901_20111031_Asia_0.5deg.nc
MONTHLY_BASELINE → era5_850hPa_JanJul_monthlymeans_1981_2020_Asia_0.5deg.nc


# 10. ดาวน์โหลด teaching data จาก GitHub

In [6]:
# CELL 6 — Download teaching NetCDF files
FORCE_DOWNLOAD = False
download_rows = []

for dataset_key, filename in TEACHING_FILES.items():
    url = f"{RAW_BASE}/{filename}"
    destination = SOURCE_GITHUB_DIR / filename

    if destination.exists() and destination.stat().st_size > 0 and not FORCE_DOWNLOAD:
        status = "ALREADY_AVAILABLE"
        print(f"{dataset_key}: already available")
    else:
        print(f"{dataset_key}: downloading ...")
        response = requests.get(
            url,
            timeout=180,
            stream=True,
            headers={"User-Agent": "Teaching-ERA5-Reanalysis-GIS"},
        )
        response.raise_for_status()

        with destination.open("wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

        if destination.stat().st_size == 0:
            raise RuntimeError(f"Downloaded file is empty: {filename}")

        status = "DOWNLOADED"

    download_rows.append({
        "dataset_key": dataset_key,
        "filename": filename,
        "status": status,
        "size_MB": destination.stat().st_size / (1024**2),
        "local_path": str(destination),
    })

DOWNLOAD_REPORT = pd.DataFrame(download_rows)
display(DOWNLOAD_REPORT)

EVENT: downloading ...
SPATIOTEMPORAL: downloading ...
MONTHLY_BASELINE: downloading ...


,dataset_key,filename,status,size_MB,local_path
0,EVENT,era5_noul_20111002_05_SEAsia_pressure_levels_0...,DOWNLOADED,10.045524,/content/drive/MyDrive/Teaching_ERA5_Reanalysi...
1,SPATIOTEMPORAL,era5_u850_6hourly_20110901_20111031_Asia_0.5de...,DOWNLOADED,8.921192,/content/drive/MyDrive/Teaching_ERA5_Reanalysi...
2,MONTHLY_BASELINE,era5_850hPa_JanJul_monthlymeans_1981_2020_Asia...,DOWNLOADED,22.253628,/content/drive/MyDrive/Teaching_ERA5_Reanalysi...


# 11. SHA256 checksum

Checksum เป็น digital fingerprint ของไฟล์

```text
file bytes
   ↓
hash function
   ↓
fixed-length fingerprint
```

In [7]:
# CELL 7 — Local checksum audit
def sha256sum(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

checksum_rows = []

for dataset_key, filename in TEACHING_FILES.items():
    path = SOURCE_GITHUB_DIR / filename
    checksum_rows.append({
        "dataset_key": dataset_key,
        "filename": filename,
        "size_MB": path.stat().st_size / (1024**2),
        "sha256": sha256sum(path),
    })

CHECKSUM_AUDIT = pd.DataFrame(checksum_rows)
display(CHECKSUM_AUDIT)

CHECKSUM_CSV = METADATA_DIR / "00_local_file_checksum_audit.csv"
CHECKSUM_AUDIT.to_csv(CHECKSUM_CSV, index=False)

print("Saved:", CHECKSUM_CSV)

,dataset_key,filename,size_MB,sha256
0,EVENT,era5_noul_20111002_05_SEAsia_pressure_levels_0...,10.045524,ab46627c04643de7067d953479340b0b1fabb6aed06cd6...
1,SPATIOTEMPORAL,era5_u850_6hourly_20110901_20111031_Asia_0.5de...,8.921192,b92be54c89c3af16c3723f8a889b489cb23841d4543aca...
2,MONTHLY_BASELINE,era5_850hPa_JanJul_monthlymeans_1981_2020_Asia...,22.253628,5fa75fbb58eeed45106436f07552b9bca0a2345e69e292...


Saved: /content/drive/MyDrive/Teaching_ERA5_Reanalysis_GIS/02_metadata/00_local_file_checksum_audit.csv


# 12. เปิด NetCDF ด้วย Xarray

In [8]:
# CELL 8 — Open datasets
DATASETS = {}

for dataset_key, filename in TEACHING_FILES.items():
    path = SOURCE_GITHUB_DIR / filename
    ds = xr.open_dataset(path, decode_times=True)
    DATASETS[dataset_key] = ds

    print("\n" + "="*70)
    print(dataset_key)
    print(ds)


EVENT
<xarray.Dataset> Size: 14MB
Dimensions:    (time: 4, level: 8, latitude: 100, longitude: 140)
Coordinates:
  * time       (time) datetime64[ns] 32B 2011-10-02 2011-10-03 ... 2011-10-05
  * level      (level) int32 32B 200 250 300 500 700 850 925 1000
  * latitude   (latitude) float32 400B 44.88 44.38 43.88 ... -4.125 -4.625
  * longitude  (longitude) float32 560B 60.12 60.62 61.12 ... 128.6 129.1 129.6
Data variables:
    d          (time, level, latitude, longitude) float32 2MB ...
    z          (time, level, latitude, longitude) float32 2MB ...
    r          (time, level, latitude, longitude) float32 2MB ...
    q          (time, level, latitude, longitude) float32 2MB ...
    t          (time, level, latitude, longitude) float32 2MB ...
    u          (time, level, latitude, longitude) float32 2MB ...
    v          (time, level, latitude, longitude) float32 2MB ...
    vo         (time, level, latitude, longitude) float32 2MB ...
Attributes: (12/17)
    Conventions:       

# 13. Dataset vs DataArray

```text
Dataset
→ container ที่มีหลาย variables

DataArray
→ ตัวแปรหนึ่งตัวพร้อม dimensions + coordinates + attributes
```

In [9]:
# CELL 9 — Dataset and DataArray demonstration
event_ds = DATASETS["EVENT"]

print("Dataset type:", type(event_ds))
print("Variables:", list(event_ds.data_vars))

temperature = event_ds["t"]

print("\nTemperature type:", type(temperature))
print(temperature)

Dataset type: <class 'xarray.core.dataset.Dataset'>
Variables: ['d', 'z', 'r', 'q', 't', 'u', 'v', 'vo']

Temperature type: <class 'xarray.core.dataarray.DataArray'>
<xarray.DataArray 't' (time: 4, level: 8, latitude: 100, longitude: 140)> Size: 2MB
[448000 values with dtype=float32]
Coordinates:
  * time       (time) datetime64[ns] 32B 2011-10-02 2011-10-03 ... 2011-10-05
  * level      (level) int32 32B 200 250 300 500 700 850 925 1000
  * latitude   (latitude) float32 400B 44.88 44.38 43.88 ... -4.125 -4.625
  * longitude  (longitude) float32 560B 60.12 60.62 61.12 ... 128.6 129.1 129.6
Attributes:
    units:          K
    long_name:      Temperature
    standard_name:  air_temperature


# 14. Structural audit

In [10]:
# CELL 10 — Structural audit
structure_rows = []

for key, ds in DATASETS.items():
    structure_rows.extend([
        {
            "dataset": key,
            "property": "dimensions",
            "value": json.dumps({k: int(v) for k, v in ds.sizes.items()}),
        },
        {
            "dataset": key,
            "property": "coordinates",
            "value": ",".join(ds.coords),
        },
        {
            "dataset": key,
            "property": "variables",
            "value": ",".join(ds.data_vars),
        },
    ])

STRUCTURE_AUDIT = pd.DataFrame(structure_rows)
display(STRUCTURE_AUDIT)

,dataset,property,value
0,EVENT,dimensions,"{""time"": 4, ""level"": 8, ""latitude"": 100, ""long..."
1,EVENT,coordinates,"longitude,latitude,level,time"
2,EVENT,variables,"d,z,r,q,t,u,v,vo"
3,SPATIOTEMPORAL,dimensions,"{""time"": 244, ""latitude"": 50, ""longitude"": 240}"
4,SPATIOTEMPORAL,coordinates,"longitude,latitude,time,source_pressure_level_hPa"
5,SPATIOTEMPORAL,variables,u
6,MONTHLY_BASELINE,dimensions,"{""time"": 80, ""latitude"": 110, ""longitude"": 160}"
7,MONTHLY_BASELINE,coordinates,"longitude,latitude,time,source_pressure_level_hPa"
8,MONTHLY_BASELINE,variables,"z,r,t,u,v,vo"


# 15. Time audit

In [11]:
# CELL 11 — Time audit
time_rows = []

for key, ds in DATASETS.items():
    times = pd.DatetimeIndex(pd.to_datetime(ds["time"].values))

    if len(times) > 1:
        delta_hours = (
            np.diff(times.values)
            .astype("timedelta64[s]")
            .astype(float)
            / 3600.0
        )
        median_step_hours = float(np.nanmedian(delta_hours))
    else:
        median_step_hours = np.nan

    time_rows.append({
        "dataset": key,
        "n_times": len(times),
        "time_start": str(times.min()),
        "time_end": str(times.max()),
        "median_time_step_hours": median_step_hours,
        "months_present": ",".join(str(v) for v in sorted(times.month.unique())),
    })

TIME_AUDIT = pd.DataFrame(time_rows)
display(TIME_AUDIT)

,dataset,n_times,time_start,time_end,median_time_step_hours,months_present
0,EVENT,4,2011-10-02 00:00:00,2011-10-05 00:00:00,24.0,10
1,SPATIOTEMPORAL,244,2011-09-01 00:00:00,2011-10-31 18:00:00,6.0,"9,10"
2,MONTHLY_BASELINE,80,1981-01-01 00:00:00,2020-07-01 00:00:00,4368.0,"1,7"


# 16. ตีความ time axis

```text
EVENT
4 times
2–5 Oct 2011
24 h interval
→ four daily 00-UTC snapshots

SPATIOTEMPORAL
244 times
Sep–Oct 2011
6 h interval

MONTHLY_BASELINE
80 times
1981–2020
January + July
```

> `24-h spacing ≠ daily mean`

# 17. Pressure-level audit

In [12]:
# CELL 12 — Pressure audit
pressure_rows = []

for key, ds in DATASETS.items():
    if "level" in ds.coords:
        levels = np.atleast_1d(ds["level"].values)
        coord_type = "dimension_coordinate"
        level_text = ",".join(str(v) for v in levels.tolist())

    elif "source_pressure_level_hPa" in ds.coords:
        coord_type = "scalar_metadata_coordinate"
        level_text = str(float(ds["source_pressure_level_hPa"].values))

    else:
        coord_type = "not_explicitly_available"
        level_text = ""

    pressure_rows.append({
        "dataset": key,
        "pressure_coordinate_type": coord_type,
        "levels_hPa": level_text,
    })

PRESSURE_AUDIT = pd.DataFrame(pressure_rows)
display(PRESSURE_AUDIT)

,dataset,pressure_coordinate_type,levels_hPa
0,EVENT,dimension_coordinate,"200,250,300,500,700,850,925,1000"
1,SPATIOTEMPORAL,scalar_metadata_coordinate,850.0
2,MONTHLY_BASELINE,scalar_metadata_coordinate,850.0


# 18. Pressure coordinates ใน course

EVENT มีหลาย pressure levels:

```text
200, 250, 300, 500, 700, 850, 925, 1000 hPa
```

SPATIOTEMPORAL และ MONTHLY_BASELINE เป็น single-level archive ที่ 850 hPa

# 19. Spatial audit

In [13]:
# CELL 13 — Spatial audit
def infer_step(values):
    values = np.asarray(values, dtype=float)
    if values.size < 2:
        return np.nan
    return float(np.nanmedian(np.abs(np.diff(values))))

spatial_rows = []

for key, ds in DATASETS.items():
    lat = ds["latitude"]
    lon = ds["longitude"]

    spatial_rows.append({
        "dataset": key,
        "latitude_min": float(lat.min()),
        "latitude_max": float(lat.max()),
        "longitude_min": float(lon.min()),
        "longitude_max": float(lon.max()),
        "latitude_step_deg": infer_step(lat.values),
        "longitude_step_deg": infer_step(lon.values),
        "latitude_order": (
            "increasing"
            if float(lat.values[-1]) > float(lat.values[0])
            else "decreasing"
        ),
        "longitude_order": (
            "increasing"
            if float(lon.values[-1]) > float(lon.values[0])
            else "decreasing"
        ),
    })

SPATIAL_AUDIT = pd.DataFrame(spatial_rows)
display(SPATIAL_AUDIT)

,dataset,latitude_min,latitude_max,longitude_min,longitude_max,latitude_step_deg,longitude_step_deg,latitude_order,longitude_order
0,EVENT,-4.625,44.875,60.125,129.625,0.5,0.5,decreasing,increasing
1,SPATIOTEMPORAL,5.375,29.875,40.125,159.625,0.5,0.5,decreasing,increasing
2,MONTHLY_BASELINE,-9.625,44.875,60.125,139.625,0.5,0.5,decreasing,increasing


# 20. Grid spacing ≠ physical resolution

```text
Source grid spacing
≠
Teaching grid spacing
≠
Effective physical resolution
```

# 21. Variable metadata inventory

In [14]:
# CELL 14 — Variable metadata inventory
variable_rows = []

for key, ds in DATASETS.items():
    for variable_name in ds.data_vars:
        variable = ds[variable_name]

        variable_rows.append({
            "dataset": key,
            "variable": variable_name,
            "long_name": variable.attrs.get("long_name", ""),
            "standard_name": variable.attrs.get("standard_name", ""),
            "units": variable.attrs.get("units", ""),
            "dims": ",".join(variable.dims),
            "dtype": str(variable.dtype),
        })

VARIABLE_AUDIT = pd.DataFrame(variable_rows)
display(VARIABLE_AUDIT)

,dataset,variable,long_name,standard_name,units,dims,dtype
0,EVENT,d,Divergence,divergence_of_wind,s**-1,"time,level,latitude,longitude",float32
1,EVENT,z,Geopotential,geopotential,m**2 s**-2,"time,level,latitude,longitude",float32
2,EVENT,r,Relative humidity,relative_humidity,%,"time,level,latitude,longitude",float32
3,EVENT,q,Specific humidity,specific_humidity,kg kg**-1,"time,level,latitude,longitude",float32
4,EVENT,t,Temperature,air_temperature,K,"time,level,latitude,longitude",float32
5,EVENT,u,U component of wind,eastward_wind,m s**-1,"time,level,latitude,longitude",float32
6,EVENT,v,V component of wind,northward_wind,m s**-1,"time,level,latitude,longitude",float32
7,EVENT,vo,Vorticity (relative),atmosphere_relative_vorticity,s**-1,"time,level,latitude,longitude",float32
8,SPATIOTEMPORAL,u,U component of wind,eastward_wind,m s**-1,"time,latitude,longitude",float32
9,MONTHLY_BASELINE,z,Geopotential,geopotential,m**2 s**-2,"time,latitude,longitude",float32


# 22. ตัวแปรสำคัญ

| Variable | Meaning | Typical unit |
|---|---|---|
| `t` | Temperature | K |
| `q` | Specific humidity | kg kg⁻¹ |
| `r` | Relative humidity | % |
| `u` | Zonal wind | m s⁻¹ |
| `v` | Meridional wind | m s⁻¹ |
| `z` | Geopotential | m² s⁻² |
| `vo` | Relative vorticity | s⁻¹ |
| `d` | Divergence | s⁻¹ |

Wind speed:

$$
V=\sqrt{u^2+v^2}
$$

Geopotential height:

$$
H=\frac{\Phi}{g_0}
$$

โดย $g_0 \approx 9.80665\ \mathrm{m\,s^{-2}}$

# 23. Provenance metadata audit

In [15]:
# CELL 15 — Provenance audit
metadata_rows = []

for key, ds in DATASETS.items():
    metadata_rows.append({
        "dataset": key,
        "dataset_key_attribute": ds.attrs.get("dataset_key", ""),
        "source_type": ds.attrs.get("source_type", ""),
        "source_file": ds.attrs.get("source_file", ""),
        "processing_description": ds.attrs.get("processing_description", ""),
        "teaching_purpose": ds.attrs.get("teaching_purpose", ""),
        "source_grid_deg": ds.attrs.get("source_horizontal_grid_spacing_deg", ""),
        "teaching_grid_deg": ds.attrs.get("teaching_horizontal_grid_spacing_deg", ""),
    })

PROVENANCE_AUDIT = pd.DataFrame(metadata_rows)
display(PROVENANCE_AUDIT)

,dataset,dataset_key_attribute,source_type,source_file,processing_description,teaching_purpose,source_grid_deg,teaching_grid_deg
0,EVENT,EVENT,ERA5 atmospheric reanalysis,era5_daily_2_5oct_2011_globe.nc,Spatial subset to Southeast Asia teaching doma...,"NetCDF/Xarray, pressure-level structure, wind,...",0.25,0.5
1,SPATIOTEMPORAL,SPATIOTEMPORAL,ERA5 atmospheric reanalysis,era5_hourly4_sep_oct_2011_globe.nc,Spatial subset to tropical Asia; retained sour...,Time-longitude and time-latitude Hovmoller ana...,0.25,0.5
2,MONTHLY_BASELINE,MONTHLY_BASELINE,ERA5 atmospheric reanalysis,era5_monthly_mean_globe.nc,Spatial subset to South/Southeast Asia teachin...,Monthly-mean archive for teaching climatologic...,0.25,0.5


# 24. Reproducibility

```text
Data
+
Metadata
+
Processing record
+
Code
=
Reproducible analysis
```

# 25. Course data audit

In [16]:
# CELL 16 — Course data audit
COURSE_AUDIT = pd.DataFrame([
    {
        "dataset": "EVENT",
        "role": "Pressure-level and synoptic analysis",
        "expected_use": "Notebooks 01–05, 09",
    },
    {
        "dataset": "SPATIOTEMPORAL",
        "role": "6-hourly 850-hPa zonal-wind spatiotemporal analysis",
        "expected_use": "Notebook 06",
    },
    {
        "dataset": "MONTHLY_BASELINE",
        "role": "Jan/Jul monthly means for climatology and anomaly",
        "expected_use": "Notebooks 07–08",
    },
])

display(COURSE_AUDIT)

COURSE_AUDIT_CSV = METADATA_DIR / "00_course_data_audit.csv"
COURSE_AUDIT.to_csv(COURSE_AUDIT_CSV, index=False)

print("Saved:", COURSE_AUDIT_CSV)

,dataset,role,expected_use
0,EVENT,Pressure-level and synoptic analysis,"Notebooks 01–05, 09"
1,SPATIOTEMPORAL,6-hourly 850-hPa zonal-wind spatiotemporal ana...,Notebook 06
2,MONTHLY_BASELINE,Jan/Jul monthly means for climatology and anomaly,Notebooks 07–08


Saved: /content/drive/MyDrive/Teaching_ERA5_Reanalysis_GIS/02_metadata/00_course_data_audit.csv


# 26. Final readiness check

In [17]:
# CELL 17 — Final readiness
readiness_rows = []

def add_check(item, passed, note=""):
    readiness_rows.append({
        "item": item,
        "status": "PASS" if passed else "FAIL",
        "note": note,
    })

add_check(
    "All three teaching files available",
    all((SOURCE_GITHUB_DIR / filename).exists() for filename in TEACHING_FILES.values()),
)

add_check(
    "All three NetCDF files opened with Xarray",
    len(DATASETS) == 3,
)

add_check(
    "EVENT has pressure-level dimension",
    "level" in DATASETS["EVENT"].coords,
)

add_check(
    "EVENT contains core meteorological variables",
    set(["t", "q", "r", "u", "v", "z", "vo", "d"]).issubset(
        set(DATASETS["EVENT"].data_vars)
    ),
)

spatial_step = TIME_AUDIT.loc[
    TIME_AUDIT["dataset"] == "SPATIOTEMPORAL",
    "median_time_step_hours",
].iloc[0]

add_check(
    "SPATIOTEMPORAL is 6-hourly",
    bool(np.isclose(spatial_step, 6.0)),
)

monthly_months = set(
    TIME_AUDIT.loc[
        TIME_AUDIT["dataset"] == "MONTHLY_BASELINE",
        "months_present",
    ].iloc[0].split(",")
)

add_check(
    "MONTHLY_BASELINE contains January and July only",
    monthly_months == {"1", "7"},
)

add_check("Course audit exported", COURSE_AUDIT_CSV.exists())
add_check("Checksum audit exported", CHECKSUM_CSV.exists())

READINESS = pd.DataFrame(readiness_rows)
display(READINESS)

ALL_READY = bool((READINESS["status"] == "PASS").all())

READINESS_CSV = METADATA_DIR / "00_readiness_check.csv"
READINESS.to_csv(READINESS_CSV, index=False)

print()

if ALL_READY:
    print("COURSE DATA READY")
    print("Next: Notebook 01 — NetCDF & Xarray Fundamentals")
else:
    print("COURSE DATA REQUIRE REVIEW")

,item,status,note
0,All three teaching files available,PASS,
1,All three NetCDF files opened with Xarray,PASS,
2,EVENT has pressure-level dimension,PASS,
3,EVENT contains core meteorological variables,PASS,
4,SPATIOTEMPORAL is 6-hourly,PASS,
5,MONTHLY_BASELINE contains January and July only,PASS,
6,Course audit exported,PASS,
7,Checksum audit exported,PASS,



COURSE DATA READY
Next: Notebook 01 — NetCDF & Xarray Fundamentals


# 27. สิ่งที่ต้องจำ

```text
ERA5 ≠ direct observation
Dataset ≠ DataArray
Dimension ≠ Coordinate
Coordinate ≠ Variable
Grid spacing ≠ physical resolution
00-UTC snapshot ≠ daily mean
Monthly mean ≠ climatological mean
850 hPa ≠ above-ground atmosphere everywhere
```

Course flow:

```text
00  Download & Data Audit
 ↓
01  NetCDF & Xarray Fundamentals
 ↓
02  Spatial Subsetting & Cartopy
 ↓
03  Pressure-Level Atmospheric Structure
 ↓
04  Wind & Vector Analysis
 ↓
05  Synoptic Weather Maps & Dynamics
 ↓
06  Hovmöller & Spatiotemporal Analysis
 ↓
07  Climatology
 ↓
08  Anomaly
 ↓
09  Integrated Noul 2011 Case Study
```

# Exercise

1. EVENT dataset มี dimensions อะไรบ้าง?
2. เหตุใด EVENT จึงไม่ควรเรียกว่า daily mean?
3. SPATIOTEMPORAL มี temporal interval กี่ชั่วโมง?
4. MONTHLY_BASELINE มีเดือนอะไรบ้าง?
5. เหตุใด MONTHLY_BASELINE ยังไม่ใช่ climatology?
6. `z` แตกต่างจาก geopotential height อย่างไร?
7. `u` และ `v` ต่างจาก wind speed อย่างไร?
8. เหตุใด 0.5° teaching grid จึงไม่ใช่ original ERA5 resolution?
9. Dataset ใดเหมาะสำหรับ Hovmöller มากที่สุด?
10. Dataset ใดเหมาะสำหรับศึกษาบรรยากาศหลาย pressure levels?